In [1]:
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import numpy as np

In [2]:
adata = sc.read_h5ad('/home/workspace/2025-mm-project/manuscript-figures/inputs/scrna_objects/final-pbmc-raw.h5ad')

In [3]:
adata

AnnData object with n_obs × n_vars = 5477370 × 31915
    obs: 'batch_id', 'cell_name', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'seurat_pbmc_type', 'seurat_pbmc_type_score', 'umap_1', 'umap_2', 'well_id', 'aifi_l1', 'aifi_l2', 'aifi_l3', 'predicted_doublet', 'doublet_score', 'sample.sampleKitGuid', 'sample.visitDetails', 'sample.visitName', 'sample.drawDate', 'sample.daysSinceFirstVisit', 'sample.diseaseStatesRecordedAtVisit', 'subject.biologicalSex', 'subject.birthYear', 'subject.ethnicity', 'subject.partnerCode', 'subject.race', 'subject.subjectGuid', 'specimen.specimenGuid', 'cohort.cohortGuid', 'manual.time_stamp', 'tissue', 'manual.response', 'manual.response_type', 'manual.extracted_name', 'manual.batch_id', 'manual.category', 'manual.treatment_dara', 'manual.flu_response', 'aifi_label_l1', 'aifi_celltype_l1', 'aifi_label_l2', 'aifi_celltype_l2', 'aifi_plot_l2', 'aifi_label_l3', 'aif

In [4]:
adata.shape

(5477370, 31915)

In [5]:
adata.X.dtype

dtype('uint16')

In [6]:
if sp.issparse(adata.X):
    adata.X = adata.X.tocsr().astype('float32')
else:
    adata.X = sp.csr_matrix(adata.X.astype('float32'))

In [7]:
adata.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 8713755905 stored elements and shape (5477370, 31915)>

In [10]:
adata.obs['log_umis'] = np.log10(adata.obs['n_umis'])
adata.obs['log_genes'] = np.log10(adata.obs['n_genes'])

In [11]:
responses = {
    'flu_responder': 'Responder',
    'flu_non_responder': 'Non-Responder',
    'None': 'Not Tested'
}
adata.obs['response'] = [responses[r] for r in adata.obs['manual.flu_response']]

In [12]:
keep_obs = {
    'batch_id': 'Batch ID',
    'log_umis': 'Log10(N Gene UMIs)',
    'log_genes': 'Log10(N Genes)',
    'subject.subjectGuid': 'Subject ID',
    'subject.biologicalSex': 'Biological Sex',
    'subject.age': 'Age',
    'subject.cmv': 'CMV Status',
    'response': 'Flu Vaccine Response',
    'label.visitDetails': 'Visit Timepoint',
    'aifi_label_l1': 'AIFI_L1',
    'aifi_label_l2': 'AIFI_L2',
    'aifi_label_l3': 'AIFI_L3'
}

obs = adata.obs.copy()
obs = obs[keep_obs.keys()]
obs = obs.rename(keep_obs, axis = 1)
obs['Age'] = obs['Age'].astype('uint16')

In [13]:
obs.shape

(5477370, 12)

In [14]:
obs.dtypes

Batch ID                category
Log10(N Gene UMIs)       float64
Log10(N Genes)           float32
Subject ID              category
Biological Sex          category
Age                       uint16
CMV Status              category
Flu Vaccine Response      object
Visit Timepoint         category
AIFI_L1                 category
AIFI_L2                 category
AIFI_L3                 category
dtype: object

In [15]:
adata.obs = obs

In [16]:
%%time
adata.write_zarr(store='adata.zarr', chunks=(5_000, adata.n_vars))

CPU times: user 1min 57s, sys: 28.3 s, total: 2min 26s
Wall time: 1min 22s


In [18]:
!du -sh adata.zarr/

20G	adata.zarr/


In [19]:
!du -sh /home/workspace/2025-mm-project/manuscript-figures/inputs/scrna_objects/final-pbmc-raw.h5ad

83G	/home/workspace/2025-mm-project/manuscript-figures/inputs/scrna_objects/final-pbmc-raw.h5ad
